In [ ]:
import os

In [ ]:
torch.version

In [ ]:
import torchvision

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

True
12.8


In [ ]:
import os
import random
import json

def create_training_pairs(sentence, overlap_range=(2, 5)):
    """Create training pairs with JSON-formatted outputs."""
    words = sentence.strip().split()
    if len(words) < 8:
        return []
    
    pairs = []
    for _ in range(2):
        overlap = random.randint(*overlap_range)
        split_point = random.randint(len(words) // 3, 2 * len(words) // 3)
        
        chunk1 = " ".join(words[:split_point + overlap])
        chunk2 = " ".join(words[split_point:])
        
        if random.random() < 0.3:
            chunk2 = add_whisper_variations(chunk2)
        
        pairs.append({
            "instruction": f"""Merge these overlapping transcripts:
1: {chunk1}
2: {chunk2}

Respond ONLY with valid JSON in this format:
{{"merged_output": "your merged text here"}}""",
            "output": f'{{"merged_output": "{sentence.strip()}"}}'
        })
    
    return pairs

def add_whisper_variations(text):
    """Add realistic Whisper-style inconsistencies."""
    variations = [
        (lambda t: t.lower()),  # All lowercase
        (lambda t: t.replace(",", "")),  # Remove commas
        (lambda t: t.replace(".", "")),  # Remove periods
    ]
    
    if random.random() < 0.5:
        variation = random.choice(variations)
        text = variation(text)
    
    return text

def load_sentences_from_directory(directory):
    """Load all sentences from text files in directory."""
    sentences = []
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            filepath = os.path.join(directory, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        # Remove the ID prefix (everything before first space after the ID)
                        parts = line.split(" ", 1)
                        if len(parts) > 1:
                            sentence = parts[1]
                        else:
                            sentence = line
                        sentences.append(sentence)
    return sentences

def generate_training_data(input_dir, output_path):
    """Generate training data and save to JSON."""
    sentences = load_sentences_from_directory(input_dir)
    print(f"Loaded {len(sentences)} sentences")
    
    all_pairs = []
    for sentence in sentences:
        pairs = create_training_pairs(sentence)
        all_pairs.extend(pairs)
    
    print(f"Generated {len(all_pairs)} training pairs")
    
    # Shuffle the pairs
    random.shuffle(all_pairs)
    
    # Save to JSON
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_pairs, f, indent=2)
    
    print(f"Saved to {output_path}")
    return all_pairs


In [ ]:
os.getcwd()

In [ ]:
sentences = load_sentences_from_directory("/dev/whisper_stream_proj/training_data")

In [ ]:
training_data = generate_training_data("./training_data", "./processed_training_data/training_pairs.json")

In [ ]:
for pair in training_data[:3]:
    print("=" * 50)
    print(pair["instruction"])
    print("-" * 20)
    print(pair["output"])

In [ ]:
from unsloth import FastLanguageModel
import torch
import json
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

In [ ]:
os.getcwd()

In [ ]:
with open("./processed_training_data/training_pairs.json", "r", encoding="utf-8") as f:
    training_data = json.load(f)

print(f"Loaded {len(training_data)} training pairs")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B",
    max_seq_length=2048,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
def format_prompt(example):
    return [f"""### Instruction:
{example['instruction']}
### Response:
{example['output']}"""]

In [ ]:
dataset = Dataset.from_list(training_data)


In [ ]:
from trl import SFTConfig, SFTTrainer


In [ ]:

# Format for training
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    }

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_prompt)

# Train
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="outputs",
        optim="adamw_8bit",
    ),
)

trainer.train()

# Save for Ollama
model.save_pretrained_gguf("whisper_stitcher", tokenizer, quantization_method="q4_k_m")

In [ ]:
os.getcwd()

In [ ]:
import ollama

In [ ]:
from pydantic import BaseModel

class merged_response(BaseModel):
  merged_output: str 

In [ ]:
section_a = "Wow that cat has such soft and fluffy fur,"
section_b = "fluffy fur, I really want to pet it"


llm_message = '''Merge these overlapping transcripts:
    1: ''' + section_a + '''
    2: ''' + section_b

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [134]:
for chunk in ollama.generate(model="whisper_stitcher_1", prompt=llm_message, stream=True):
    print(chunk['response'], end='', flush=True)



HOW MUCH RANDAL ARRARED AT SANDALABLE CHILDDOCKANDRANDALARRATED AT SANDALARY POPULARCHILDPLAYING RANDAL ARRALLED AT SANDALARY POPULARCHILDPLAYING AND SHALL OBSTALY RANDAL ARRALLED AT SANDALARY POPULARCHILDPLAYING AND SHALL OBSTALY RANDAL ARRALED AT SANDALARY POPULARCHILDPLAYING AND SHALL OBSTALY RANDAL ARRALED AT SANDALARY RANDAL ARRALLED AT SANDALARY RANDAL ARRALLED AT SANDALARY RANDAL ARRALLED AT SANDALARY RANDAL ARRALLED AT SANDALARY RANDAL ARRALLED AT SANDALARY CHARMING SYDNEY BENNYDECKAND SHALL OBSTALY RANDAL ARRALED AT SANDALARY CHARMING SYDNEY BENNYDECKAND SHALL OBSTALY RANDAL ARRALED AT SANDALARY CHARMING SYDNEY BENNYDECKAND SHALL OBSTALY RANDAL ARRALED AT SANDALARY CHARMING SYDNEY BENNYDECKAND SHALL OBSTALY RANDAL ARRALLED AT SANDALARY CHARMING SYDNEY BENNYDECKAND SHALL OBSTALY RANDAL ARRALLED AT SANDALARY CHARMING SYDNEY BENNYDECKAND SHALL OBSTALY RANDAL ARRALED AT SANDALARY CHARMING SYDNEY BENNYDECKAND SHALL OBSTALY RANDAL ARRALED AT SANDALARY CHARMING SYDNEY BENNYDECKAND 

KeyboardInterrupt: 

In [ ]:
response = ollama.chat(model='whisper_stitcher_1', format = merged_response.model_json_schema(), messages=[
        {'role': 'system', 'content': '''You merge overlapping transcript fragments. The end of transcript 1 overlaps with the start of transcript 2. Remove the duplication and return ONLY the merged text.
        
        Rules:
        - Output ONLY the merged text
        - No explanations, notes, or commentary
        - If there is no clear overlap, return transcript 2 unchanged
        
        Example:
        1: The cat sat on the mat
        2: on the mat and then fell asleep
        Output: The cat sat on the mat and then fell asleep'''}
        ,
        {'role': 'user', 'content': llm_message}
    ])


merged_response.model_validate_json(response.message.content).merged_output

In [ ]:
response